# Week 28, Day 3: Neurons and glia: what a cell type is

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w028/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Clustering units by spike width and rate

200 simulated units have two features, spike width (ms) and log firing rate; 40 are narrow-spiking. Standardise each feature, then write `kmeans(X, k=2, iters=50)` (from each of 10 random starts, local rng seed 2, alternate assigning each point to its nearest centre and moving each centre to its points' mean; keep the run with the smallest total squared distance). Store the result in `labels`, and the accuracy against the true classes, taking the better of the two ways of naming the clusters, in `accuracy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(3)
narrow = np.arange(200) < 40
width = np.where(narrow, 0.3, 0.65) + np.where(narrow, 0.05, 0.09) * r.normal(size=200)
lograte = np.where(narrow, np.log(15), np.log(3)) + 0.6 * r.normal(size=200)
X = np.c_[width, lograte]
Xs = (X - X.mean(0)) / X.std(0)

def kmeans(X, k=2, iters=50, restarts=10):
    best, best_cost = None, np.inf
    rk = np.random.default_rng(2)
    for _ in range(restarts):                     # several random starts; keep the tightest clustering
        centres = X[rk.choice(len(X), k, replace=False)]
        for _ in range(iters):
            d = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(-1)   # points x centres
            lab = ...
            centres = ...
        cost = d[np.arange(len(X)), lab].sum()
        if cost < best_cost: best, best_cost = lab, cost
    return best

labels = kmeans(Xs)
accuracy = max(np.mean(labels == narrow), np.mean(labels != narrow))
print(f"accuracy {accuracy:.3f}")
plt.scatter(width, np.exp(lograte), c=labels, s=8); plt.yscale("log"); plt.xlabel("width (ms)"); plt.ylabel("rate (Hz)"); plt.show()

In [ ]:
import numpy as np
def test_labels():
    "labels assigns every unit to cluster 0 or 1"
    assert labels.shape == (200,) and set(np.unique(labels)) <= {0, 1}

def test_accuracy():
    "after matching cluster names to the true classes, accuracy is above 90%"
    assert accuracy > 0.9

test_labels()
test_accuracy()
print("All checks passed")

### Exercise: PCA and clustering of single-cell expression

300 cells of three types (100 each) express 50 genes; each type has 5 marker genes raised by 3 units, on top of noise. Log-transform and centre the expression, take the top five principal components into `Z`, cluster with your k-means (k = 3), and compute `type_accuracy`, the agreement with the true types under the best of the 6 possible matchings of cluster names.

In [ ]:
from itertools import permutations
rg = np.random.default_rng(18)
types = np.repeat([0, 1, 2], 100)
expr = rg.gamma(2.0, 1.0, (300, 50))
for t in range(3):
    expr[types == t, t * 5:(t + 1) * 5] += 3.0                  # marker genes, like Pvalb, Sst and Vip
Y = np.log1p(expr); Y = Y - Y.mean(0)
Vt = np.linalg.svd(Y, full_matrices=False)[2]
Z = ...
clusters = kmeans(Z, k=3)
type_accuracy = max(np.mean(np.array(p)[clusters] == types) for p in permutations(range(3)))
print(f"type recovery {type_accuracy:.3f}")
plt.scatter(Z[:, 0], Z[:, 1], c=clusters, s=8); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

In [ ]:
import numpy as np
def test_pcs():
    "Z holds each cell's scores on the top five principal components"
    assert Z.shape == (300, 5)

def test_types():
    "the best matching of clusters to true types is over 95% accurate"
    assert type_accuracy > 0.95

test_pcs()
test_types()
print("All checks passed")

### Exercise: A network that obeys Dale's principle

Build a 500-neuron connectivity matrix W (row = target, column = source) with 400 excitatory and 100 inhibitory neurons, connection probability 0.1 (local rng seed 5), excitatory weight w = 0.1 and inhibitory weight −g·w. Choose g so that, on average, each neuron's excitatory and inhibitory inputs cancel.

In [ ]:
rd = np.random.default_rng(5)
N, NE = 500, 400; NI = N - NE; w = 0.1
mask = rd.random((N, N)) < 0.1
g = ...                                          # inhibitory strength relative to excitatory
W = np.zeros((N, N))
W[:, :NE] = ...
W[:, NE:] = ...
print(f"g = {g};  mean excitatory input {W[:, :NE].sum(1).mean():.2f}, mean inhibitory {W[:, NE:].sum(1).mean():.2f}, net {W.sum(1).mean():.3f}")

In [ ]:
import numpy as np
def test_signs():
    "every excitatory column is non-negative and every inhibitory column non-positive"
    assert np.all(W[:, :NE] >= 0) and np.all(W[:, NE:] <= 0)

def test_balance():
    "with g = NE/NI times the excitatory weight, the mean input to each neuron is close to zero"
    assert abs(W.sum(1).mean()) < 0.05 * W[:, :NE].sum(1).mean()

test_signs()
test_balance()
print("All checks passed")